In [1]:
import matplotlib
matplotlib.use("Agg")
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
import h5py

import importlib
import os
import sys
import re
import glob

In [2]:
path_openket = "//home//ultrxvioletx//GitHub//openket"
if path_openket not in sys.path:
    sys.path.append(path_openket)
# elimina el caché de func.py para que no lea los datos anteriores
if 'func' in sys.modules:
    del sys.modules['func']
if os.path.exists('func.py'):
    os.remove('func.py')

from openket.core.diracobject import Ket, Bra, Operator, AnnihilationOperator, CreationOperator
from openket.core.evolution import build_ode, sym2num, init_state
from openket.core.metrics import dag, comm, ptrace, trace, normalize, sub_qexpr, op2dict, qmatrix

import style
importlib.reload(style)
from style import *
set_style()

2026-03-31 19:05:38,828 - openket - INFO - openket v0.1.0 initialized successfully.


##### funciones auxiliares

In [3]:
def convergencia(tiempo, fotones, tol=1e-3, porcentaje=0.1):
    npoints = int(len(tiempo) * porcentaje)
    if npoints < 2:
        return False, np.nan # no hay suficientes puntos
    fotones = fotones[-npoints:]
    tiempo = tiempo[-npoints:]
    df = np.gradient(fotones, tiempo)
    df_mean = np.mean(np.abs(df))
    return df_mean < tol, df_mean

In [4]:
def procesafile(at, lvl, folder, barrido="scan"):
    global detunings, Nss, non_converged, Probs, niveles, attrs
    detunings = []
    Probs = [[] for _ in np.arange(lvl**at)]
    Nss = []
    attrs = {}
    
    non_converged = []
    files = glob.glob(os.path.join(path,'*.h5'))
    for i,file in enumerate(files):
        with h5py.File(file, 'r') as f:
            dataset = os.path.basename(file).replace('.h5', '')
            if i == 0:
                attrs = dict(f[dataset].attrs)
            
            detuning = f[dataset].attrs[barrido]
            tt = f[dataset].attrs['t']
            niveles = f[dataset].attrs['probs_label']
            tiempo = np.linspace(float(tt[0]), float(tt[1]), int(tt[2]))
    
            if dataset not in f:
                print(f"  no se encontró el dataset '{dataset}' en el archivo. Saltando.")
                continue

            detunings.append(detuning)
            rho = f[dataset][:]
            lvl_prob = [rho[:, i] for i in np.arange(lvl**at)] # las primeras n columnas son las probabilidades
            N_expect = rho[:, lvl**at] # la última columna es el valor medio de fotones en la cavidad
            N_expect = np.real(N_expect)

            # promediamos el último 25% de los puntos
            npoints = int(rho.shape[0] * 0.25)
            Nss.append(np.mean(N_expect[-npoints:]))
            for i,ele in enumerate(lvl_prob):
                Probs[i].append(np.mean(ele[-npoints:]))
            
            is_converged, derivative = convergencia(tiempo, N_expect)
            if not is_converged:
                non_converged.append([detuning, np.mean(N_expect[-npoints:])])
                
    # ordena por detuning para mejorar la gráfica
    detunings = np.array(detunings)
    sorti = np.argsort(detunings)
    
    detunings = detunings[sorti]
    Nss = np.array(Nss)[sorti]
    Probs = [np.real(np.array(p))[sorti] for p in Probs]

In [30]:
def picos(x,y):
    peaks, _ = find_peaks(y, prominence=0.01)
    top2 = np.argsort(np.array(y)[peaks])[-2:]
    x1, x2 = sorted(np.array(x)[peaks[top2]])
        
    return x1, x2

## solo átomo

In [55]:
at,lvl = 1,4
file = os.path.join("asaf",f"{at}at{lvl}lvl",f"{at}at{lvl}lvl.h5")
with h5py.File(file, 'r') as f:
    dataset = os.path.basename(file).replace('.h5', '')
    attrs = dict(f[dataset].attrs)
    
    tt = f[dataset].attrs['t']
    niveles = f[dataset].attrs['probs_label']
    tiempo = np.linspace(float(tt[0]), float(tt[1]), int(tt[2]))

    rho = f[dataset][:]
    lvl_prob = [rho[:, i] for i in np.arange(lvl**at)] # las primeras n columnas son las probabilidades
plvls = ["e", "s"]
step = 20
for j,plvl in enumerate(plvls):
    pattern_str = plvl.replace("_", ".")
    pattern_str = pattern_str[:at] # recorta al largo del dataset
    pattern = re.compile(f"^{pattern_str}$")
    indices = [i for i, lbl in enumerate(niveles) if pattern.search(lbl)]
    prob = np.sum([lvl_prob[i] for i in indices], axis=0)
    
    plt.plot(tiempo[::step], prob[::step], "s-", label=rf"${latex["P"]}{plvl}$", markersize=5, linewidth=1, color=colores["estados"][j])
plt.axhline(0, color="gray", linestyle="--", linewidth=1)
plt.axhline(1, color="gray", linestyle="--", linewidth=1)
plt.xlabel(r"tiempo $\kappa ^{-1}$")
plt.ylabel(r"Probabilidad de excitación")
plt.legend()
ax = plt.gca()
format_ax(ax, xstep=50)
plt.savefig(f"../figuras/1at4lvl.png")
plt.close()

In [7]:
attrs

{'detuning_c': np.float64(-99.0),
 'detuning_p': np.float64(100.000625),
 'gamma_12': np.float64(0.0),
 'gamma_e': np.float64(0.0),
 'gamma_s': np.float64(0.0),
 'kappa': np.float64(1.0),
 'omega_DR': np.float64(0.0),
 'omega_EE': np.float64(0.0),
 'probs_label': array(['g', 'e', 's', 'p'], dtype=object),
 'rabi_c': np.float64(20.0),
 'rabi_p': np.float64(0.5),
 'scan': np.float64(0.0),
 'stark_g': np.float64(0.000625),
 'stark_s': np.float64(1.0),
 't': array(['0', '200.0', '1000'], dtype=object)}

## átomo + cavidad

In [42]:
at,lvl = 1,4
label = "cavidad"
path = os.path.join("asaf",f"{at}at{lvl}lvl_{label}")

procesafile(at, lvl, path)

plt.plot(detunings, Nss, "s-", markersize=5, linewidth=1, color=colores["fotones"])
if non_converged:
    dtn = [ele[0] for ele in non_converged]
    nmean = [ele[1] for ele in non_converged]
    plt.scatter(dtn, nmean, color="red")
plt.axvline(1, color="gray", linestyle="--", linewidth=1, alpha=0.5)
x1,x2 = picos(detunings,Nss)
ax = plt.gca()
segmento(ax, x1, x2, y=max(Nss))
#plt.xlabel(rf"${latex["Dpa"]}$")
plt.ylabel(rf"Número medio de fotones ${latex["Nss"]}$")
format_ax(ax, xstep=1, ystep=0.2, ymax=1)
plt.savefig(f"../figuras/1at4lvl_fotones.png")
plt.close()
fdist_1at = x2-x1

plvls = ["e", "s"]
for j,plvl in enumerate(plvls):
    pattern_str = plvl.replace("_", ".")
    pattern_str = pattern_str[:at] # recorta al largo del dataset
    pattern = re.compile(f"^{pattern_str}$")
    indices = [i for i, lbl in enumerate(niveles) if pattern.search(lbl)]
    prob = np.sum([Probs[i] for i in indices], axis=0)
    
    plt.plot(detunings, prob, "s-", label=rf"${latex["P"]}{plvl}$", markersize=5, linewidth=1, color=colores["estados"][j])
plt.axvline(1, color="gray", linestyle="--", linewidth=1, alpha=0.5)
x1, x2 = picos(detunings,prob)
ax = plt.gca()
segmento(ax, x1, x2, y=max(prob))
plt.xlabel(rf"${latex["scan"]}$")
plt.ylabel(r"Probabilidad de excitación")
ax = plt.gca()
format_ax(ax, xstep=1, ystep=0.2, ymax=1)
plt.legend()
plt.savefig(f"../figuras/1at4lvl_excitado.png")
plt.close()
edist_1at = x2-x1

In [10]:
attrs

{'detuning_c': np.float64(-99.0),
 'detuning_p': np.float64(106.455125),
 'gamma_12': np.float64(0.0),
 'gamma_e': np.float64(1.0),
 'gamma_s': np.float64(0.0),
 'idx': np.int32(53),
 'kappa': np.float64(1.0),
 'omega_DR': np.float64(0.0),
 'omega_EE': np.float64(0.0),
 'probs_label': array(['g', 'e', 's', 'p'], dtype=object),
 'rabi_c': np.float64(20.0),
 'rabi_p': np.float64(0.5),
 'scan': np.float64(6.4545),
 'stark_g': np.float64(0.000625),
 'stark_s': np.float64(1.0),
 't': array(['0', '200.0', '1000'], dtype=object)}

## 2 átomos + cavidad

In [44]:
at,lvl = 2,4
label = "cavidad"
path = os.path.join("asaf",f"{at}at{lvl}lvl_{label}")

procesafile(at, lvl, path)

plt.plot(detunings, Nss, "s-", markersize=5, linewidth=1, color=colores["fotones"])
if non_converged:
    dtn = [ele[0] for ele in non_converged]
    nmean = [ele[1] for ele in non_converged]
    plt.scatter(dtn, nmean, color="red")
plt.axvline(1, color="gray", linestyle="--", linewidth=1, alpha=0.5)
x1, x2 = picos(detunings,Nss)
ax = plt.gca()
segmento(ax, x1, x2, y=max(Nss))
#plt.xlabel(rf"${latex["Dpa"]}$")
plt.ylabel(rf"Número medio de fotones ${latex["Nss"]}$")
ax = plt.gca()
format_ax(ax, xstep=1, ystep=0.2, ymax=1)
plt.savefig(f"../figuras/2at4lvl_fotones0.png")
plt.close()
fdist_2at = x2-x1

plvls = ["s.", "ss"]
labels = ["P_{gs}+P_{sg}", "P_{ss}"]
for j,plvl in enumerate(plvls):
    pattern_str = plvl.replace("_", ".")
    pattern_str = pattern_str[:at] # recorta al largo del dataset
    pattern = re.compile(f"^{pattern_str}$")
    indices = [i for i, lbl in enumerate(niveles) if pattern.search(lbl)]
    prob = np.sum([Probs[i] for i in indices], axis=0)
    if j==0: prob_s = prob
    
    plt.plot(detunings, prob, "s-", label=rf"${labels[j]}$", markersize=5, linewidth=1, color=colores["estados"][j+1])
plt.axvline(1, color="gray", linestyle="--", linewidth=1, alpha=0.5)
x1, x2 = picos(detunings,prob_s)
ax = plt.gca()
segmento(ax, x1, x2, y=max(prob_s))
plt.xlabel(rf"${latex["Dpa"]}$")
plt.ylabel(r"Probabilidad de excitación")
ax = plt.gca()
format_ax(ax, xstep=1, ystep=0.2, ymax=1)
plt.legend()
plt.savefig(f"../figuras/2at4lvl_excitado0.png")
plt.close()
edist_2at = x2-x1

In [13]:
attrs

{'detuning_c': np.float64(-99.0),
 'detuning_p': np.float64(101.364225),
 'gamma_12': np.float64(0.0),
 'gamma_e': np.float64(1.0),
 'gamma_s': np.float64(0.0),
 'idx': np.int32(26),
 'kappa': np.float64(1.0),
 'omega_DR': np.float64(0.0),
 'omega_EE': np.float64(0.0),
 'probs_label': array(['gg', 'ge', 'gs', 'gp', 'eg', 'ee', 'es', 'ep', 'sg', 'se', 'ss',
        'sp', 'pg', 'pe', 'ps', 'pp'], dtype=object),
 'rabi_c': np.float64(20.0),
 'rabi_p': np.float64(0.5),
 'scan': np.float64(1.3636),
 'stark_g': np.float64(0.000625),
 'stark_s': np.float64(1.0),
 't': array(['0', '200.0', '1000'], dtype=object)}

In [51]:
print("1at=",fdist_1at, "2at=",fdist_2at, "factor=",fdist_2at/fdist_1at)
print(edist_2at/edist_1at)

1at= 3.0909 2at= 5.6 factor= 1.8117700346177488
1.837538673083534


## 2 átomos + cavidad + bloqueo

In [59]:
at,lvl = 2,4
label = "bloqueo50"
path = os.path.join("asaf",f"{at}at{lvl}lvl_{label}")

procesafile(at, lvl, path)

plt.plot(detunings, Nss, "s-", markersize=5, linewidth=1, color=colores["fotones"])
if non_converged:
    dtn = [ele[0] for ele in non_converged]
    nmean = [ele[1] for ele in non_converged]
    plt.scatter(dtn, nmean, color="red")
plt.axvline(1, color="gray", linestyle="--", linewidth=1, alpha=0.5)
x1, x2 = picos(detunings,Nss)
ax = plt.gca()
segmento(ax, x1, x2, y=max(Nss))
plt.xlabel(rf"${latex["Dpa"]}$")
plt.ylabel(rf"Número medio de fotones ${latex["Nss"]}$")
ax = plt.gca()
format_ax(ax, xstep=1, ystep=0.2, ymax=1)
plt.savefig(f"../figuras/2at4lvl_fotones.png")
plt.close()
fdist_2atb = x2-x1

plvls = ["s.", "ss"]
labels = ["s", "ss"]
for j,plvl in enumerate(plvls):
    pattern_str = plvl.replace("_", ".")
    pattern_str = pattern_str[:at] # recorta al largo del dataset
    pattern = re.compile(f"^{pattern_str}$")
    indices = [i for i, lbl in enumerate(niveles) if pattern.search(lbl)]
    prob = np.sum([Probs[i] for i in indices], axis=0)
    
    plt.plot(detunings, prob, "s-", label=rf"${latex["P"]}{plvl}$", markersize=5, linewidth=1, color=colores["estados"][j])
plt.axvline(1, color="gray", linestyle="--", linewidth=1, alpha=0.5)
x1, x2 = picos(detunings,prob_s)
ax = plt.gca()
segmento(ax, x1, x2, y=max(prob_s))
plt.xlabel(rf"${latex["scan"]}$")
plt.ylabel(r"Probabilidad de excitación")
ax = plt.gca()
format_ax(ax, xstep=1, ystep=0.2, ymax=1)
plt.legend()
plt.savefig(f"../figuras/2at4lvl_excitado.png")
plt.close()

In [ ]:
attrs

##### aux